<a href="https://colab.research.google.com/github/mmbc560/GUIA2/blob/main/Identifying_Supply_Chain_Vulnerabilities.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ================================================================
# REACTIVE VS PROACTIVE SUPPLY CHAIN MANAGEMENT
# ================================================================
#
# OBJETIVO DEL EJERCICIO
#
# Comparar dos formas diferentes de gestionar inventarios y riesgo:
#
# SCENARIO 1 - REACTIVE MANAGEMENT
# La empresa espera a que el problema ocurra para actuar.
#
# Ejemplos:
# - Reacciona cuando ya existe un stockout.
# - Compra de emergencia.
# - Cambia de proveedor después del incumplimiento.
# - Mantiene inventarios sin considerar riesgo.
#
# SCENARIO 2 - PROACTIVE MANAGEMENT
# La empresa utiliza datos y señales de riesgo antes de que ocurra
# el problema.
#
# Ejemplos:
# - Ajusta el safety stock.
# - Diversifica proveedores.
# - Anticipa retrasos.
# - Reordena antes de llegar a niveles críticos.
#
# ================================================================


# ================================================================
# 1. IMPORTAR LIBRERÍAS
# ================================================================

# pandas permite trabajar con tablas.
import pandas as pd

# numpy permite realizar cálculos y generar datos.
import numpy as np

# matplotlib permite crear gráficas.
import matplotlib.pyplot as plt


# ================================================================
# 2. CREAR LOS DATOS DEL CASO
# ================================================================

# Utilizamos una semilla para obtener siempre los mismos resultados.
np.random.seed(10)


# Vamos a trabajar con 12 productos.
products = [
    "Product A",
    "Product B",
    "Product C",
    "Product D",
    "Product E",
    "Product F",
    "Product G",
    "Product H",
    "Product I",
    "Product J",
    "Product K",
    "Product L"
]


# Creamos una base ficticia.
data = pd.DataFrame({

    "Product": products,

    # Demanda promedio mensual
    "Monthly_Demand": [
        900, 1200, 700, 1500,
        1000, 850, 1300, 600,
        1100, 950, 1400, 750
    ],

    # Inventario disponible inicialmente
    "Initial_Inventory": [
        1000, 900, 800, 1100,
        950, 700, 1000, 650,
        850, 900, 950, 700
    ],

    # Lead time promedio del proveedor en días
    "Lead_Time": [
        12, 28, 15, 40,
        22, 35, 18, 10,
        32, 20, 45, 16
    ],

    # Variabilidad de la demanda
    #
    # 0.10 significa aproximadamente 10% de variabilidad.
    "Demand_Variability": [
        0.10, 0.22, 0.12, 0.30,
        0.18, 0.27, 0.15, 0.08,
        0.25, 0.14, 0.32, 0.11
    ],

    # Riesgo del proveedor en escala de 0 a 100.
    #
    # Mayor valor = proveedor más vulnerable.
    "Supplier_Risk": [
        20, 55, 25, 80,
        45, 70, 30, 15,
        65, 35, 90, 22
    ]
})


# ================================================================
# 3. CREAR EL ESCENARIO REACTIVO
# ================================================================
#
# En una gestión reactiva:
#
# - La empresa usa poco safety stock.
# - No ajusta inventario según riesgo del proveedor.
# - Tiene mayor probabilidad de stockout.
# - Puede necesitar compras urgentes.
# ================================================================


# Safety stock muy sencillo:
# solamente 10% de la demanda.
data["Reactive_Safety_Stock"] = (
    data["Monthly_Demand"] * 0.10
)


# Generamos una demanda real.
#
# Esta demanda cambia debido a la incertidumbre del mercado.
data["Actual_Demand"] = (

    data["Monthly_Demand"] *

    (
        1 +
        np.random.normal(
            0,
            data["Demand_Variability"]
        )
    )
)


# Aseguramos que la demanda no sea negativa.
data["Actual_Demand"] = data["Actual_Demand"].clip(lower=0)


# Inventario disponible en el escenario reactivo.
data["Reactive_Available_Inventory"] = (

    data["Initial_Inventory"] +
    data["Reactive_Safety_Stock"]
)


# Calculamos el inventario final.
#
# Si es negativo, significa que ocurrió un stockout.
data["Reactive_Final_Inventory"] = (

    data["Reactive_Available_Inventory"] -
    data["Actual_Demand"]
)


# Calculamos las unidades faltantes.
#
# Si el inventario final es negativo,
# tomamos su valor absoluto.
data["Reactive_Stockout"] = (

    -data["Reactive_Final_Inventory"]
).clip(lower=0)


# Indicamos si hubo stockout.
data["Reactive_Stockout_Flag"] = (

    data["Reactive_Stockout"] > 0
).astype(int)


# ================================================================
# 4. CREAR EL ESCENARIO PROACTIVO
# ================================================================
#
# En el escenario proactivo:
#
# - El safety stock depende de la variabilidad.
# - También considera el lead time.
# - Incluye una señal de riesgo del proveedor.
# - La empresa protege mejor los productos críticos.
# ================================================================


# Creamos un factor de riesgo del proveedor.
#
# Convertimos la escala 0-100 a 0-1.
data["Supplier_Risk_Factor"] = (

    data["Supplier_Risk"] / 100
)


# Calculamos un safety stock más inteligente.
#
# Consideramos:
#
# 1. Demanda
# 2. Variabilidad
# 3. Lead time
# 4. Riesgo del proveedor
#
# La fórmula es simplificada y tiene fines educativos.
data["Proactive_Safety_Stock"] = (

    data["Monthly_Demand"]

    *

    (
        0.10

        +

        data["Demand_Variability"] * 0.70

        +

        (data["Lead_Time"] / 60) * 0.20

        +

        data["Supplier_Risk_Factor"] * 0.20
    )
)


# Inventario disponible con gestión proactiva.
data["Proactive_Available_Inventory"] = (

    data["Initial_Inventory"] +
    data["Proactive_Safety_Stock"]
)


# Inventario final.
data["Proactive_Final_Inventory"] = (

    data["Proactive_Available_Inventory"] -
    data["Actual_Demand"]
)


# Unidades faltantes.
data["Proactive_Stockout"] = (

    -data["Proactive_Final_Inventory"]
).clip(lower=0)


# Indicador de stockout.
data["Proactive_Stockout_Flag"] = (

    data["Proactive_Stockout"] > 0
).astype(int)


# ================================================================
# 5. CALCULAR COSTOS
# ================================================================
#
# Ahora vamos a traducir los resultados operativos a dinero.
#
# Supuestos:
#
# Holding Cost:
# costo de mantener una unidad en inventario.
#
# Stockout Cost:
# costo de no poder satisfacer una unidad de demanda.
#
# Emergency Cost:
# costo adicional generado por una compra urgente.
# ================================================================


holding_cost = 4

stockout_cost = 18

emergency_cost = 10


# ------------------------------------------------
# COSTOS DEL ESCENARIO REACTIVO
# ------------------------------------------------

data["Reactive_Holding_Cost"] = (

    data["Reactive_Final_Inventory"]
    .clip(lower=0)
    *
    holding_cost
)


data["Reactive_Stockout_Cost"] = (

    data["Reactive_Stockout"]
    *
    stockout_cost
)


# En el escenario reactivo asumimos que cualquier stockout
# genera compras de emergencia.
data["Reactive_Emergency_Cost"] = (

    data["Reactive_Stockout"]
    *
    emergency_cost
)


data["Reactive_Total_Cost"] = (

    data["Reactive_Holding_Cost"]
    +
    data["Reactive_Stockout_Cost"]
    +
    data["Reactive_Emergency_Cost"]
)


# ------------------------------------------------
# COSTOS DEL ESCENARIO PROACTIVO
# ------------------------------------------------

data["Proactive_Holding_Cost"] = (

    data["Proactive_Final_Inventory"]
    .clip(lower=0)
    *
    holding_cost
)


data["Proactive_Stockout_Cost"] = (

    data["Proactive_Stockout"]
    *
    stockout_cost
)


# En el escenario proactivo se reduce la necesidad de compras urgentes.
#
# Supondremos que solo 30% del faltante requiere compra de emergencia.
data["Proactive_Emergency_Cost"] = (

    data["Proactive_Stockout"]
    *
    emergency_cost
    *
    0.30
)


data["Proactive_Total_Cost"] = (

    data["Proactive_Holding_Cost"]
    +
    data["Proactive_Stockout_Cost"]
    +
    data["Proactive_Emergency_Cost"]
)


# ================================================================
# 6. CALCULAR INDICADORES GENERALES
# ================================================================


reactive_stockouts = data[
    "Reactive_Stockout_Flag"
].sum()


proactive_stockouts = data[
    "Proactive_Stockout_Flag"
].sum()


reactive_total_cost = data[
    "Reactive_Total_Cost"
].sum()


proactive_total_cost = data[
    "Proactive_Total_Cost"
].sum()


# Service Level:
#
# porcentaje de productos que no tuvieron stockout.
reactive_service_level = (

    1 -
    reactive_stockouts / len(data)
) * 100


proactive_service_level = (

    1 -
    proactive_stockouts / len(data)
) * 100


# ================================================================
# GRÁFICA 1
# SAFETY STOCK COMPARISON
# ================================================================
#
# Esta gráfica muestra la principal diferencia conceptual:
#
# REACTIVE:
# todos los productos reciben una protección muy similar.
#
# PROACTIVE:
# los productos con mayor incertidumbre y riesgo
# reciben mayor protección.
# ================================================================


x = np.arange(len(data))

width = 0.38


plt.figure(figsize=(14,7))


plt.bar(
    x - width/2,
    data["Reactive_Safety_Stock"],
    width,
    label="Reactive",
    color="#E74C3C"
)


plt.bar(
    x + width/2,
    data["Proactive_Safety_Stock"],
    width,
    label="Proactive",
    color="#2ECC71"
)


plt.title(
    "Safety Stock Strategy: Reactive vs Proactive",
    fontsize=17,
    fontweight="bold"
)


plt.xlabel(
    "Products"
)


plt.ylabel(
    "Safety Stock Units"
)


plt.xticks(
    x,
    data["Product"],
    rotation=45
)


plt.legend()


plt.grid(
    axis="y",
    alpha=0.20
)


plt.tight_layout()

plt.show()


# ================================================================
# GRÁFICA 2
# STOCKOUT COMPARISON
# ================================================================
#
# Esta gráfica permite ver dónde la gestión reactiva
# genera faltantes que podrían haber sido prevenidos.
# ================================================================


plt.figure(figsize=(14,7))


plt.bar(
    x - width/2,
    data["Reactive_Stockout"],
    width,
    label="Reactive",
    color="#C0392B"
)


plt.bar(
    x + width/2,
    data["Proactive_Stockout"],
    width,
    label="Proactive",
    color="#27AE60"
)


plt.title(
    "Stockout Exposure: Reactive vs Proactive",
    fontsize=17,
    fontweight="bold"
)


plt.xlabel(
    "Products"
)


plt.ylabel(
    "Stockout Units"
)


plt.xticks(
    x,
    data["Product"],
    rotation=45
)


plt.legend()


plt.grid(
    axis="y",
    alpha=0.20
)


plt.tight_layout()

plt.show()


# ================================================================
# GRÁFICA 3
# TOTAL COST BY PRODUCT
# ================================================================
#
# Esta gráfica introduce un concepto importante:
#
# mayor inventario NO necesariamente significa mayor costo total.
#
# Una estrategia proactiva puede mantener más safety stock,
# pero evitar:
#
# - ventas perdidas
# - compras urgentes
# - interrupciones
# ================================================================


plt.figure(figsize=(14,7))


plt.bar(
    x - width/2,
    data["Reactive_Total_Cost"],
    width,
    label="Reactive",
    color="#E67E22"
)


plt.bar(
    x + width/2,
    data["Proactive_Total_Cost"],
    width,
    label="Proactive",
    color="#3498DB"
)


plt.title(
    "Total Operational Cost: Reactive vs Proactive",
    fontsize=17,
    fontweight="bold"
)


plt.xlabel(
    "Products"
)


plt.ylabel(
    "Estimated Cost"
)


plt.xticks(
    x,
    data["Product"],
    rotation=45
)


plt.legend()


plt.grid(
    axis="y",
    alpha=0.20
)


plt.tight_layout()

plt.show()


# ================================================================
# GRÁFICA 4
# SERVICE LEVEL COMPARISON
# ================================================================
#
# Aquí resumimos el desempeño total de ambos escenarios.
# ================================================================


scenarios = [
    "Reactive",
    "Proactive"
]


service_levels = [
    reactive_service_level,
    proactive_service_level
]


plt.figure(figsize=(9,6))


bars = plt.bar(
    scenarios,
    service_levels,
    color=[
        "#E74C3C",
        "#2ECC71"
    ]
)


plt.title(
    "Service Level Comparison",
    fontsize=17,
    fontweight="bold"
)


plt.ylabel(
    "Service Level (%)"
)


plt.ylim(
    0,
    105
)


# Mostramos el porcentaje encima de cada barra.
for bar in bars:

    height = bar.get_height()

    plt.text(
        bar.get_x() + bar.get_width()/2,
        height + 2,
        f"{height:.1f}%",
        ha="center",
        fontweight="bold",
        fontsize=12
    )


plt.grid(
    axis="y",
    alpha=0.20
)


plt.tight_layout()

plt.show()


# ================================================================
# GRÁFICA 5
# TOTAL SUPPLY CHAIN COST
# ================================================================
#
# Comparamos el costo completo de ambos escenarios.
# ================================================================


total_costs = [

    reactive_total_cost,

    proactive_total_cost
]


plt.figure(figsize=(9,6))


bars = plt.bar(
    scenarios,
    total_costs,
    color=[
        "#C0392B",
        "#2980B9"
    ]
)


plt.title(
    "Total Supply Chain Cost",
    fontsize=17,
    fontweight="bold"
)


plt.ylabel(
    "Total Estimated Cost"
)


# Mostramos el valor de cada barra.
for bar in bars:

    height = bar.get_height()

    plt.text(
        bar.get_x() + bar.get_width()/2,
        height + 300,
        f"${height:,.0f}",
        ha="center",
        fontweight="bold",
        fontsize=12
    )


plt.grid(
    axis="y",
    alpha=0.20
)


plt.tight_layout()

plt.show()


# ================================================================
# GRÁFICA 6
# VULNERABILITY MAP
# ================================================================
#
# Esta gráfica permite ver por qué algunos productos
# necesitan una estrategia más preventiva.
#
# X = Lead Time
# Y = Supplier Risk
# Tamaño de burbuja = Monthly Demand
#
# Los productos ubicados arriba a la derecha son especialmente
# vulnerables porque combinan:
#
# - proveedor de alto riesgo
# - lead time largo
# ================================================================


bubble_size = (

    data["Monthly_Demand"] /
    data["Monthly_Demand"].max()
) * 1500


# Creamos colores de acuerdo con el riesgo.
risk_colors = []


for risk in data["Supplier_Risk"]:

    if risk < 35:

        risk_colors.append("#2ECC71")

    elif risk < 65:

        risk_colors.append("#F1C40F")

    else:

        risk_colors.append("#E74C3C")


plt.figure(figsize=(12,7))


plt.scatter(

    data["Lead_Time"],

    data["Supplier_Risk"],

    s=bubble_size,

    c=risk_colors,

    alpha=0.70,

    edgecolors="black"
)


# Colocamos el nombre de cada producto.
for i, row in data.iterrows():

    plt.annotate(

        row["Product"],

        (
            row["Lead_Time"],
            row["Supplier_Risk"]
        ),

        xytext=(5,5),

        textcoords="offset points"
    )


plt.axvline(

    30,

    color="#34495E",

    linestyle="--"
)


plt.axhline(

    60,

    color="#34495E",

    linestyle="--"
)


plt.text(

    37,

    85,

    "HIGH VULNERABILITY\nPROACTIVE ACTION",

    color="#E74C3C",

    fontsize=12,

    fontweight="bold"
)


plt.title(

    "Supply Chain Vulnerability Map",

    fontsize=17,

    fontweight="bold"
)


plt.xlabel(
    "Supplier Lead Time (days)"
)


plt.ylabel(
    "Supplier Risk Score"
)


plt.grid(
    alpha=0.20
)


plt.tight_layout()

plt.show()


# ================================================================
# 7. RESUMEN EJECUTIVO
# ================================================================


print("\n" + "=" * 70)

print(
    "REACTIVE VS PROACTIVE SUPPLY CHAIN MANAGEMENT"
)

print("=" * 70)


print("\nSCENARIO 1 - REACTIVE MANAGEMENT")

print("-" * 50)

print(
    f"Products with Stockout: {reactive_stockouts}"
)

print(
    f"Service Level: {reactive_service_level:.1f}%"
)

print(
    f"Total Estimated Cost: ${reactive_total_cost:,.0f}"
)


print("\nSCENARIO 2 - PROACTIVE MANAGEMENT")

print("-" * 50)

print(
    f"Products with Stockout: {proactive_stockouts}"
)

print(
    f"Service Level: {proactive_service_level:.1f}%"
)

print(
    f"Total Estimated Cost: ${proactive_total_cost:,.0f}"
)


# ================================================================
# 8. MOSTRAR TABLA FINAL
# ================================================================

print("\nPRODUCT LEVEL ANALYSIS\n")


summary = data[
    [
        "Product",
        "Monthly_Demand",
        "Lead_Time",
        "Supplier_Risk",
        "Reactive_Safety_Stock",
        "Proactive_Safety_Stock",
        "Reactive_Stockout",
        "Proactive_Stockout",
        "Reactive_Total_Cost",
        "Proactive_Total_Cost"
    ]
]


print(
    summary.round(1).to_string(index=False)
)


# ================================================================
# 9. DECISIÓN GERENCIAL
# ================================================================

print("\n" + "=" * 70)

print(
    "MANAGEMENT QUESTION"
)

print("=" * 70)


print(
    """
If you were the Supply Chain Manager:

1. Which products require proactive management first?

2. Is holding additional inventory always more expensive?

3. Which products combine high supplier risk and long lead times?

4. Where would you increase safety stock?

5. Where would you consider a second supplier?

6. Which scenario creates the best balance between
   service level, inventory and total cost?
"""
)